In [ ]:
import pandas as pd
import sqlite3
import numpy as np

db_path = r"../data/database/faers_2025.db"
conn = sqlite3.connect(db_path)

In [ ]:
df_drug = pd.read_sql_query("SELECT primaryid, final_drug_name FROM drug_clean WHERE role_cod = 'PS'", conn)
df_reac = pd.read_sql_query("SELECT primaryid, pt AS symptom FROM reac_clean", conn)
conn.close()

In [ ]:
df_drug['final_drug_name'] = df_drug['final_drug_name'].astype(str).str.upper().str.strip()
df_reac['symptom'] = df_reac['symptom'].astype(str).str.upper().str.strip()

In [ ]:
df_master = df_drug.merge(df_reac, on='primaryid', how='inner')
# Calculate total number of unique reports in the entire dataset
total_reports = df_master['primaryid'].nunique()
print(f"Total Unique Reports (N): {total_reports:,}")

In [ ]:
# Count 'a': Number of reports with [Drug X] AND [Symptom Y]
pair_counts = df_master.groupby(['final_drug_name', 'symptom'])['primaryid'].nunique().reset_index(name='a')

# Count (a+b): Total reports for [Drug X] regardless of symptom
drug_totals = df_master.groupby('final_drug_name')['primaryid'].nunique().reset_index(name='drug_total')

# Count (a+c): Total reports for [Symptom Y] regardless of drug
symptom_totals = df_master.groupby('symptom')['primaryid'].nunique().reset_index(name='symptom_total')

# Merge all counts back together
df_signals = pair_counts.merge(drug_totals, on='final_drug_name', how='left')
df_signals = df_signals.merge(symptom_totals, on='symptom', how='left')

# Derive b, c, d
df_signals['b'] = df_signals['drug_total'] - df_signals['a']
df_signals['c'] = df_signals['symptom_total'] - df_signals['a']
df_signals['d'] = total_reports - (df_signals['a'] + df_signals['b'] + df_signals['c'])

In [ ]:
# Avoid division by zero
epsilon = 1e-9

# PRR Formula: (a / (a + b)) / (c / (c + d))
df_signals['PRR'] = (df_signals['a'] / (df_signals['a'] + df_signals['b'] + epsilon)) / \
                    (df_signals['c'] / (df_signals['c'] + df_signals['d'] + epsilon))

# ROR Formula: (a / b) / (c / d)
df_signals['ROR'] = (df_signals['a'] / (df_signals['b'] + epsilon)) / \
                    (df_signals['c'] / (df_signals['d'] + epsilon))

In [ ]:
# Remove infinity artifacts and ensure the symptom is not an ultra-rare typo
df_signals_clean = df_signals.replace([np.inf, -np.inf], np.nan).dropna(subset=['PRR', 'ROR'])

validated_signals = df_signals_clean[
    (df_signals_clean['a'] >= 10) &              # Minimum 10 cases with this specific drug
    (df_signals_clean['PRR'] >= 2.0) &           # PRR threshold
    (df_signals_clean['ROR'] >= 2.0) &           # ROR threshold
    (df_signals_clean['symptom_total'] >= 50)    # Symptom must exist at least 50 times globally
].copy()

# Sort by number of cases (a) and then PRR to see the most impactful signals
validated_signals = validated_signals.sort_values(by=['a', 'PRR'], ascending=[False, False])


In [ ]:
print("TOP 15 REAL DRUG-SYMPTOM SIGNALS DETECTED:")
top_15 = validated_signals[['final_drug_name', 'symptom', 'a', 'PRR', 'ROR']].head(15)
print(top_15.to_string(index=False))

# Save the cleaned signal report
validated_signals.to_csv("FDA_Real_Signals_Report.csv", index=False)